# NHANES Finite-Sample Bootstrap

How sensitive are the reported bounds and potencies to sampling noise in the
NHANES data?

Each replicate resamples the $n$ complete cases with replacement, recomputes
the empirical joint $P(Z,X,Y)$, and re-runs the full potency computation:
observational bounds plus one coupled polynomial program per candidate set,
exactly as in `../nhanes_pipeline.ipynb`. The resample is the only source of
randomness that changes across replicates.

Results are appended to `bootstrap_results.csv` after every replicate;
replicates already in that file are skipped, so the loop is resumable.

## Configuration

In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

N_REPLICAS = 50            # bootstrap replicates
BOOTSTRAP_SEED = 12345     # replicate b resamples with random_state = SEED + b
INCLUDE_POINT_EST = True   # also record the un-resampled full-data estimate
TIME_LIMIT = 300           # per-solve cap (seconds)

ROOT = Path.cwd()
SUPP_ROOT = ROOT if (ROOT / 'coupled_potency.py').exists() else ROOT.parent
sys.path.insert(0, str(SUPP_ROOT))

DATA_DIR = SUPP_ROOT / 'data'
PV_TMP = ROOT / '_pv_bootstrap_tmp.csv'      # per-replicate joint, for autobound
RESULTS_CSV = ROOT / 'bootstrap_results.csv'

from coupled_potency import evaluate_potency, nhanes_dag, observational_bounds

print(f'replicates={N_REPLICAS}  seed={BOOTSTRAP_SEED}  -> {RESULTS_CSV.name}')

replicates=50  seed=12345  -> bootstrap_results.csv


## Part 1: Load and discretize

Identical to Part 1 of `../nhanes_pipeline.ipynb`, but stopping at the
individual-level table `complete` (one row per respondent). The bootstrap
resamples its rows.

In [2]:
SPECIAL_MISSING = {7, 9, 77, 99, 777, 999}

def load_xpt(fname):
    df = pd.read_sas(str(DATA_DIR / fname))
    df.columns = [c.decode() if isinstance(c, bytes) else str(c) for c in df.columns]
    return df

demo = load_xpt('DEMO_J.xpt')
paq  = load_xpt('PAQ_J.xpt')
hiq  = load_xpt('HIQ_J.xpt')
diq  = load_xpt('DIQ_J.xpt')

merged = demo.merge(paq, on='SEQN', how='left') \
             .merge(hiq, on='SEQN', how='left') \
             .merge(diq, on='SEQN', how='left')
merged = merged[pd.to_numeric(merged['RIDAGEYR'], errors='coerce') >= 18].copy()
print(f'Adults 18+: {len(merged)}')

def clean(s):
    x = pd.to_numeric(s, errors='coerce')
    return x.mask(x.isin(SPECIAL_MISSING))

df = pd.DataFrame(index=merged.index)

# Z: Insurance (HIQ011)
hiq011 = clean(merged['HIQ011'])
df['Z'] = np.nan
df.loc[hiq011 == 1, 'Z'] = 1
df.loc[hiq011 == 2, 'Z'] = 0

# X: Physical activity (PAQ650 OR PAQ665)
paq650 = clean(merged['PAQ650'])
paq665 = clean(merged['PAQ665'])
df['X'] = np.nan
df.loc[(paq650 == 1) | (paq665 == 1), 'X'] = 1
df.loc[(paq650 == 2) & (paq665 == 2), 'X'] = 0

# Y: Diabetes (DIQ010) - Y=1 means no diabetes, Y=0 means has diabetes
diq010 = clean(merged['DIQ010'])
df['Y'] = np.nan
df.loc[diq010 == 1, 'Y'] = 0
df.loc[diq010.isin([2, 3]), 'Y'] = 1

complete = df[['Z', 'X', 'Y']].dropna().astype(int).reset_index(drop=True)
N_COMPLETE = len(complete)
print(f'Complete cases: {N_COMPLETE}')
for v in ['Z', 'X', 'Y']:
    print(f'  P({v}=1) = {complete[v].mean():.4f}')

Adults 18+: 5856
Complete cases: 5833
  P(Z=1) = 0.8471
  P(X=1) = 0.4680
  P(Y=1) = 0.8495


In [3]:
def pv_from_complete(complete_df: pd.DataFrame) -> pd.DataFrame:
    """Empirical joint P(Z,X,Y): observed cells only, exact renormalization."""
    pv = complete_df.value_counts().reset_index(name='count')
    pv['prob'] = pv['count'] / pv['count'].sum()
    diff = round(1.0 - pv['prob'].sum(), 10)
    if abs(diff) > 0:
        pv.loc[pv['count'].idxmax(), 'prob'] += diff
    return (pv[['Z', 'X', 'Y', 'prob']]
            .sort_values(['Z', 'X', 'Y']).reset_index(drop=True))

print(pv_from_complete(complete).to_string(index=False))

 Z  X  Y     prob
 0  0  0 0.009258
 0  0  1 0.080747
 0  1  0 0.004115
 0  1  1 0.058803
 1  0  0 0.092062
 1  0  1 0.349906
 1  1  0 0.045088
 1  1  1 0.360021


## Part 2: Potency for one joint distribution

Same candidates, costs and labels as Table 1.

In [4]:
XY_CELLS = [(0, 0), (0, 1), (1, 0), (1, 1)]
A3_QUERIES = [f'X(Z={z})={x}&Y(Z={z})={y}' for z in (0, 1) for x, y in XY_CELLS]

CANDIDATES = [
    ('a3_tilde', A3_QUERIES),
    ('a4_tilde', ['Y(X=0)=1&Y(X=1)=0']),
    ('a5_tilde', ['Y(X=0)=1']),
    ('a6_tilde', ['Y(X=1)=1']),
    ('{a4_tilde, a6_tilde}', ['Y(X=0)=1&Y(X=1)=0', 'Y(X=1)=1']),
    ('{a4_tilde, a5_tilde}', ['Y(X=0)=1&Y(X=1)=0', 'Y(X=0)=1']),
]
COSTS = {'a3_tilde': 1, 'a4_tilde': 1, 'a5_tilde': 2, 'a6_tilde': 2,
         '{a4_tilde, a6_tilde}': 3, '{a4_tilde, a5_tilde}': 3}

def run_all_experiments(pv_df: pd.DataFrame) -> list[dict]:
    """Observational bounds + the six candidates for one joint P(Z,X,Y)."""
    pv_df.to_csv(PV_TMP, index=False)
    dag = nhanes_dag()

    obs = observational_bounds(PV_TMP, dag=dag, time_limit=TIME_LIMIT,
                               workdir=str(ROOT))
    W_empty = obs['W_empty']
    rows = [{'candidate': 'empty', 'queries': '',
             'lb_empty': obs['lb'], 'ub_empty': obs['ub'], 'W_empty': W_empty,
             'W_a': W_empty, 'potency': 0.0, 'potency_clipped': 0.0,
             'ok': bool(obs['max_run']['ok'] and obs['min_run']['ok']),
             'termination': f"{obs['max_run']['termination']}/"
                            f"{obs['min_run']['termination']}",
             'n_vars': obs['n_vars'], 'n_constraints': obs['n_constraints'],
             'solve_seconds': obs['wall']}]
    if W_empty is None:
        return rows

    for label, queries in CANDIDATES:
        res = evaluate_potency(PV_TMP, queries, W_empty, dag=dag,
                               time_limit=TIME_LIMIT, workdir=str(ROOT))
        run, pot = res['run'], res['potency']
        rows.append({'candidate': label, 'queries': ';'.join(queries),
                     'lb_empty': obs['lb'], 'ub_empty': obs['ub'],
                     'W_empty': W_empty, 'W_a': res['W_a'], 'potency': pot,
                     'potency_clipped': (max(0.0, pot) if pot is not None else None),
                     'ok': bool(run.get('ok')),
                     'termination': run.get('termination'),
                     'n_vars': res['n_vars'], 'n_constraints': res['n_constraints'],
                     'solve_seconds': run.get('wall', run.get('seconds'))})
    return rows

COLUMNS = ['kind', 'replica', 'seed', 'n_samples', 'candidate', 'queries',
           'lb_empty', 'ub_empty', 'W_empty', 'W_a', 'potency',
           'potency_clipped', 'ok', 'termination', 'n_vars', 'n_constraints',
           'solve_seconds']

def append_rows(rows):
    out = pd.DataFrame(rows).reindex(columns=COLUMNS)
    header = not (RESULTS_CSV.exists() and RESULTS_CSV.stat().st_size > 0)
    out.to_csv(RESULTS_CSV, mode='w' if header else 'a', header=header, index=False)

def load_done():
    if not RESULTS_CSV.exists() or RESULTS_CSV.stat().st_size == 0:
        return set()
    d = pd.read_csv(RESULTS_CSV)
    return set(zip(d['kind'], d['replica'].astype(int)))

print('helpers defined')

helpers defined


## Part 3: Bootstrap loop

In [5]:
done = load_done()
t_all = time.time()

if INCLUDE_POINT_EST and ('full_data', -1) not in done:
    t0 = time.time()
    rows = run_all_experiments(pv_from_complete(complete))
    for r in rows:
        r.update({'replica': -1, 'kind': 'full_data', 'seed': None,
                  'n_samples': N_COMPLETE})
    append_rows(rows)
    done = load_done()
    print(f"point estimate: W_empty={rows[0]['W_empty']:.6f}  {time.time()-t0:.1f}s")

per_rep = []
for b in range(N_REPLICAS):
    if ('bootstrap', b) in done:
        continue
    seed = BOOTSTRAP_SEED + b
    boot = complete.sample(n=N_COMPLETE, replace=True,
                           random_state=seed).reset_index(drop=True)
    t0 = time.time()
    rows = run_all_experiments(pv_from_complete(boot))
    for r in rows:
        r.update({'replica': b, 'kind': 'bootstrap', 'seed': seed,
                  'n_samples': N_COMPLETE})
    append_rows(rows)
    per_rep.append(time.time() - t0)
    bad = [r['candidate'] for r in rows if not r['ok']]
    print(f"replica {b:>2} (seed={seed})  W_empty={rows[0]['W_empty']:.6f}"
          f"  {per_rep[-1]:5.1f}s" + (f'  FAILED: {bad}' if bad else ''))

print(f'\ncomputed {len(per_rep)} replicate(s) in {time.time()-t_all:.1f}s'
      + (f' ({np.mean(per_rep):.1f}s mean)' if per_rep else ''))
PV_TMP.unlink(missing_ok=True)

point estimate: W_empty=0.863810  2.8s


replica  0 (seed=12345)  W_empty=0.833336    4.5s


replica  1 (seed=12346)  W_empty=0.875636    2.5s


replica  2 (seed=12347)  W_empty=0.907468    2.4s


replica  3 (seed=12348)  W_empty=0.829945    2.3s


...

replica 45 (seed=12390)  W_empty=0.899579    2.9s


replica 46 (seed=12391)  W_empty=0.830513    5.0s


replica 47 (seed=12392)  W_empty=0.868821    2.9s


replica 48 (seed=12393)  W_empty=0.839102    2.8s


replica 49 (seed=12394)  W_empty=0.878312    2.8s

computed 50 replicate(s) in 150.7s (3.0s mean)


## Part 4: Aggregate

In [6]:
res = pd.read_csv(RESULTS_CSV)
boot = res[res['kind'] == 'bootstrap']
point = res[res['kind'] == 'full_data'].set_index('candidate')
order = [c for c, _ in CANDIDATES]

assert bool(res['ok'].all()), 'some solve failed'

we = boot.groupby('replica')['W_empty'].first()
print(f'replicates          : {boot["replica"].nunique()}')
print(f'W_empty  mean/std   : {we.mean():.6f} / {we.std():.6f}')
print(f'W_empty  min/max    : {we.min():.6f} / {we.max():.6f}\n')

summary = (boot.groupby('candidate')
               .agg(W_a_mean=('W_a', 'mean'),
                    pot_mean=('potency_clipped', 'mean'),
                    pot_std=('potency_clipped', 'std'),
                    n=('potency_clipped', 'size'))
               .reindex(order))
summary['pot_point_est'] = point['potency_clipped'].reindex(order)
summary['reported'] = [f'{m:.3f} +/- {s:.3f}'
                       for m, s in zip(summary['pot_mean'], summary['pot_std'])]
pd.set_option('display.float_format', lambda v: f'{v:.6f}')
print(summary.to_string())

replicates          : 50
W_empty  mean/std   : 0.859884 / 0.027709
W_empty  min/max    : 0.818011 / 0.935217

                      W_a_mean  pot_mean  pot_std   n  pot_point_est         reported
candidate                                                                            
a3_tilde              0.859884  0.000000 0.000000  50       0.000000  0.000 +/- 0.000
a4_tilde              0.421802  0.438082 0.009365  50       0.439980  0.438 +/- 0.009
a5_tilde              0.499166  0.360718 0.019231  50       0.363291  0.361 +/- 0.019
a6_tilde              0.360718  0.499166 0.011935  50       0.500519  0.499 +/- 0.012
{a4_tilde, a6_tilde}  0.360718  0.499166 0.011935  50       0.500519  0.499 +/- 0.012
{a4_tilde, a5_tilde}  0.086118  0.773766 0.023645  50       0.776365  0.774 +/- 0.024


### Optimal choice per budget, per replicate

In [7]:
TABLE1_CHOICE = {1: 'a4_tilde', 2: 'a6_tilde', 3: '{a4_tilde, a5_tilde}'}

agree = {B: 0 for B in TABLE1_CHOICE}
for b, g in boot.groupby('replica'):
    pot = dict(zip(g['candidate'], g['potency_clipped']))
    for B, expected in TABLE1_CHOICE.items():
        affordable = {k: v for k, v in pot.items() if COSTS.get(k, 99) <= B}
        if max(affordable, key=affordable.get) == expected:
            agree[B] += 1

n_rep = boot['replica'].nunique()
for B, expected in TABLE1_CHOICE.items():
    print(f'B={B}: Table 1 choice {expected:<22} '
          f'reproduced in {agree[B]}/{n_rep} replicates')

B=1: Table 1 choice a4_tilde               reproduced in 50/50 replicates
B=2: Table 1 choice a6_tilde               reproduced in 50/50 replicates
B=3: Table 1 choice {a4_tilde, a5_tilde}   reproduced in 50/50 replicates
